# Unified CONDOR/FlexDC inference + evaluation notebook

This notebook is for running the new unified prediction, optimization, and end-to-end evaluation scripts in Google Colab.

It supports all four trained model variants:

- `condor / normal`
- `condor / raw`
- `flexdc / normal`
- `flexdc / raw`

**Run/skip rule:** cells marked **Colab only** should be run in Google Colab. Cells marked **Local optional** are for reference if you copy commands back to your PC.


## 1. Environment controls — RUN in Colab

Edit the repo URLs before running. If your repositories are private, use a GitHub token or mount Drive and copy files manually.


In [ ]:
from pathlib import Path
import os

RUN_ENV = "colab"  # expected: "colab" for this notebook
WORKSPACE = Path("/content/workspace")

# TODO: replace these with your actual repo URLs and branches.
COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC"
FLEXDC_REPO_URL = "https://github.com/amenon871/FlexDC"
COMDER_BRANCH = "main"
FLEXDC_BRANCH = "main"

# Device: use cuda in Colab GPU runtime; cpu also works.
DEVICE = "cuda"

# W&B controls. Set WANDB_MODE="disabled" to turn logging off.
WANDB_ENTITY = "amenon06-boston-university"
WANDB_PROJECT = "flexdc-unified-inference"
WANDB_MODE = "online"

print("RUN_ENV:", RUN_ENV)
print("WORKSPACE:", WORKSPACE)


RUN_ENV: colab
WORKSPACE: /content/workspace


## 2. Install minimal dependencies — RUN in Colab

Do not install the whole repo requirements file unless needed. Colab already includes PyTorch in most runtimes.


In [ ]:
%pip install -q wandb pandas numpy scipy scikit-learn tqdm matplotlib tabulate

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if DEVICE == "cuda" and not torch.cuda.is_available():
    print("WARNING: DEVICE is cuda but torch.cuda.is_available() is False. Change DEVICE to 'cpu'.")


Torch: 2.11.0+cu128
CUDA available: True


## 3. Clone repositories — RUN in Colab

Skip this cell only if you already copied both repos into `/content/workspace`.


In [ ]:
!rm -rf "$WORKSPACE"
!mkdir -p "$WORKSPACE"

!git clone --branch "$COMDER_BRANCH" "$COMDER_REPO_URL" "$WORKSPACE/comder-main"
!git clone --branch "$FLEXDC_BRANCH" "$FLEXDC_REPO_URL" "$WORKSPACE/flexdc-sim"


Cloning into '/content/workspace/comder-main'...
remote: Enumerating objects: 509, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 509 (delta 0), reused 5 (delta 0), pack-reused 499 (from 2)
Receiving objects: 100% (509/509), 363.09 MiB | 23.43 MiB/s, done.
Resolving deltas: 100% (95/95), done.
Updating files: 100% (446/446), done.
Cloning into '/content/workspace/flexdc-sim'...
remote: Enumerating objects: 1274, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 1274 (delta 1), reused 9 (delta 1), pack-reused 1257 (from 1)
Receiving objects: 100% (1274/1274), 213.69 MiB | 17.90 MiB/s, done.
Resolving deltas: 100% (269/269), done.


## 4. Define paths — RUN after clone/copy

This cell makes imports work and defines where the model/data/scripts live.


In [ ]:
import sys

COMDER_ROOT = WORKSPACE / "comder-main"
FLEXDC_ROOT = WORKSPACE / "flexdc-sim"
AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
TRAIN_DIR = AM_FLEXDC_ROOT / "train"
PILOT_DIR = AM_FLEXDC_ROOT / "data" / "pilots" / "traditionaliso_newqos_pilot_flexdc_configured_objective"
MODELS_DIR = AM_FLEXDC_ROOT / "models"
RESULTS_DIR = AM_FLEXDC_ROOT / "results" / "unified_eval_runs"
DATASET_TAG = "newqos_configured_objective_v1"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

RESULTS_CSV = PILOT_DIR / "traditional_iso16_newqos_AQA_combined_grid_search_results.csv"
DIAGNOSTICS_CSV = PILOT_DIR / "traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv"

print("TRAIN_DIR:", TRAIN_DIR)
print("PILOT_DIR:", PILOT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


TRAIN_DIR: /content/workspace/comder-main/am_flexdc/train
PILOT_DIR: /content/workspace/comder-main/am_flexdc/data/pilots/traditionaliso_newqos_pilot
RESULTS_DIR: /content/workspace/comder-main/am_flexdc/results/unified_eval_runs


## 5. Optional Google Drive dataset/model copy — RUN only if data/model are not in GitHub

If the combined pilot CSVs or model checkpoints are too large for GitHub, mount Drive and copy them into the expected folders.

Skip this cell if the files are already in the repo.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# PILOT_DIR.mkdir(parents=True, exist_ok=True)
# !cp "/content/drive/MyDrive/path/to/configured_objective/traditional_iso16_newqos_AQA_combined_grid_search_results.csv" "$PILOT_DIR/"
# !cp "/content/drive/MyDrive/path/to/configured_objective/traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv" "$PILOT_DIR/"
#
# MODELS_DIR.mkdir(parents=True, exist_ok=True)
# !cp -r "/content/drive/MyDrive/path/to/models/*" "$MODELS_DIR/"


## 6. Required-file check — RUN before any inference

This catches missing scripts, datasets, or checkpoints before the long FlexDC run starts.


In [ ]:
required = [
    TRAIN_DIR / "data_center_model.py",
    TRAIN_DIR / "am_unified_training_utilities.py",
    TRAIN_DIR / "am_unified_predict_one.py",
    TRAIN_DIR / "am_unified_optimize_one.py",
    TRAIN_DIR / "am_unified_end_to_end_eval_configured_objective.py",
    RESULTS_CSV,
    DIAGNOSTICS_CSV,
    FLEXDC_ROOT / "src" / "peacsim" / "am_data_extraction_wizard.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Fix missing files before continuing.")
print("All required files found.")


Missing required files:
 - /content/workspace/comder-main/am_flexdc/data/pilots/traditionaliso_newqos_pilot/traditional_iso16_newqos_AQA_combined_grid_search_results.csv
 - /content/workspace/comder-main/am_flexdc/data/pilots/traditionaliso_newqos_pilot/traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv


FileNotFoundError: Fix missing files before continuing.

## 5b. [ALL] Validate configured objective columns

This checks that the dataset columns being used for FlexDC-normal labels have been recomputed with the configured objective parameters from the reference simulated-annealing cost configuration. It does not change the CSVs; it only validates them before training/inference.


In [ ]:
import ast
import json
import numpy as np
import pandas as pd

FLEXDC_OBJECTIVE_CONSTANTS = {
    "ctrack_psi": 1.0,
    "ctrack_mu": 10.0,
    "ctrack_gamma": 0.3,
    "qos_beta": 20.0,
    "qos_rho": 2.0,
    "qos_threshold": 0.1,
}


def _stable_softplus(x):
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _parse_probability_vector(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    if pd.isna(value):
        raise ValueError("QoS_Delay_Probabilities contains NaN")
    text = str(value).strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        parsed = ast.literal_eval(text)
    arr = np.asarray(parsed, dtype=float)
    if arr.ndim != 1:
        raise ValueError(f"Expected 1-D QoS probability vector, got {arr.shape}")
    return arr


def validate_configured_objective_columns(results_csv, diagnostics_csv, tol=1e-8):
    result_needed = {
        "Simulator_RSR_Total_Cost",
        "QoS_Delay_Probabilities",
        "Ctrack_Epsilon_90th",
        "Ctrack_Gamma",
        "Ctrack_Psi",
        "Ctrack_Mu",
        "Ctrack_Weighted_Cost",
        "Diagnostic_FlexDC_SoftPlus_QoS_Cost",
        "Diagnostic_FullPaperObjective_Cost",
    }
    diag_needed = {
        "Ctrack_Gamma",
        "Ctrack_Psi",
        "Ctrack_Mu",
        "Ctrack_Weighted_Cost",
        "Diagnostic_FlexDC_SoftPlus_QoS_Cost",
        "Diagnostic_FullPaperObjective_Cost",
    }

    res = pd.read_csv(results_csv, usecols=lambda c: c in result_needed, low_memory=False)
    diag = pd.read_csv(diagnostics_csv, usecols=lambda c: c in diag_needed, low_memory=False)

    missing_res = sorted(result_needed - set(res.columns))
    missing_diag = sorted(diag_needed - set(diag.columns))
    if missing_res or missing_diag:
        raise KeyError(f"Missing required columns. results missing={missing_res}, diagnostics missing={missing_diag}")
    if len(res) != len(diag):
        raise ValueError(f"Row-count mismatch: results={len(res)}, diagnostics={len(diag)}")

    c = FLEXDC_OBJECTIVE_CONSTANTS
    for col, expected in [
        ("Ctrack_Psi", c["ctrack_psi"]),
        ("Ctrack_Mu", c["ctrack_mu"]),
        ("Ctrack_Gamma", c["ctrack_gamma"]),
    ]:
        max_diff_res = float((res[col].astype(float) - expected).abs().max())
        max_diff_diag = float((diag[col].astype(float) - expected).abs().max())
        if max(max_diff_res, max_diff_diag) > tol:
            raise AssertionError(f"{col} does not match expected {expected}: results diff={max_diff_res}, diagnostics diff={max_diff_diag}")

    eps = res["Ctrack_Epsilon_90th"].astype(float).to_numpy()
    expected_ctrack = c["ctrack_psi"] * _stable_softplus(c["ctrack_mu"] * (eps - c["ctrack_gamma"]))

    probs = [_parse_probability_vector(v) for v in res["QoS_Delay_Probabilities"]]
    expected_qos = np.asarray([
        c["qos_beta"] * float(np.sum(_stable_softplus(c["qos_rho"] * (p - c["qos_threshold"]))))
        for p in probs
    ], dtype=float)

    expected_full = res["Simulator_RSR_Total_Cost"].astype(float).to_numpy() + expected_ctrack + expected_qos

    checks = {
        "results Ctrack reconstruction": float(np.max(np.abs(res["Ctrack_Weighted_Cost"].astype(float).to_numpy() - expected_ctrack))),
        "results QoS reconstruction": float(np.max(np.abs(res["Diagnostic_FlexDC_SoftPlus_QoS_Cost"].astype(float).to_numpy() - expected_qos))),
        "results full reconstruction": float(np.max(np.abs(res["Diagnostic_FullPaperObjective_Cost"].astype(float).to_numpy() - expected_full))),
        "results-vs-diagnostics Ctrack": float(np.max(np.abs(res["Ctrack_Weighted_Cost"].astype(float).to_numpy() - diag["Ctrack_Weighted_Cost"].astype(float).to_numpy()))),
        "results-vs-diagnostics QoS": float(np.max(np.abs(res["Diagnostic_FlexDC_SoftPlus_QoS_Cost"].astype(float).to_numpy() - diag["Diagnostic_FlexDC_SoftPlus_QoS_Cost"].astype(float).to_numpy()))),
        "results-vs-diagnostics full": float(np.max(np.abs(res["Diagnostic_FullPaperObjective_Cost"].astype(float).to_numpy() - diag["Diagnostic_FullPaperObjective_Cost"].astype(float).to_numpy()))),
    }

    bad = {name: diff for name, diff in checks.items() if diff > tol}
    if bad:
        raise AssertionError(f"Configured objective validation failed: {bad}")

    summary = pd.DataFrame([
        {"Check": name, "Max absolute difference": diff} for name, diff in checks.items()
    ])
    print("Configured FlexDC objective validation passed.")
    print("Constants:", FLEXDC_OBJECTIVE_CONSTANTS)
    print(f"Rows checked: {len(res):,}")
    display(summary)
    return summary

configured_objective_validation = validate_configured_objective_columns(RESULTS_CSV, DIAGNOSTICS_CSV)


## 7. W&B login — RUN if WANDB_MODE is online/offline

Skip if `WANDB_MODE = "disabled"`.


In [ ]:
if WANDB_MODE != "disabled":
    import wandb
    wandb.login()
else:
    print("W&B disabled.")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amenon06 (amenon06-boston-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 8. Choose model variant and checkpoint — RUN

Change these controls to test a different trained model.

For example:
- `TARGET_FAMILY="condor"`, `TARGET_MODE="normal"`
- `TARGET_FAMILY="condor"`, `TARGET_MODE="raw"`
- `TARGET_FAMILY="flexdc"`, `TARGET_MODE="normal"`
- `TARGET_FAMILY="flexdc"`, `TARGET_MODE="raw"`


In [ ]:
TARGET_FAMILY = "condor"   # "condor" or "flexdc"
TARGET_MODE = "normal"     # "normal" or "raw"
RAW_QOS_AGGREGATION = "mean"
USE_NORM_COST = "auto"     # auto means CONDOR=True, FlexDC=False
USE_NORM_PR = "true"       # should match training default

MODEL_FILE = MODELS_DIR / f"{TARGET_FAMILY}_{TARGET_MODE}" / f"am_{TARGET_FAMILY}_{TARGET_MODE}_{DATASET_TAG}_state_dict.pt"

# Override manually if your checkpoint has a different name:
# MODEL_FILE = MODELS_DIR / "condor_normal" / "your_checkpoint.pt"

if not MODEL_FILE.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {MODEL_FILE}")

print("Model variant:", TARGET_FAMILY, TARGET_MODE)
print("Model file:", MODEL_FILE)


Model variant: condor normal
Model file: /content/workspace/comder-main/am_flexdc/models/condor_normal/am_condor_normal_newqos_pilot_v1_state_dict.pt


## 9. Choose test scenario — RUN

This is the model/FlexDC context to evaluate. Starting values do not have to be optimal; they just seed gradient descent.


In [ ]:
WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5555.ini"
EXPERIMENT_CONFIG = FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini"
SERVER_COUNT = 1000
UTILIZATION = 0.80

START_PBAR = 0.4858666666666666
START_R = 0.2225185185185185
START_WEIGHTS = "0.25,0.25,0.25,0.25"

ITERATIONS = 1000
LR = 0.01

# For CONDOR normal, released example weights are 0.05,0.7,2.0.
# For FlexDC normal, auto defaults to 1,1,1 because the targets already include beta/psi weights.
OBJECTIVE_WEIGHTS = "auto"

print("Workload config:", WORKLOAD_CONFIG)
print("Experiment config:", EXPERIMENT_CONFIG)


Workload config: /content/workspace/flexdc-sim/configs/workload/W2-short-qos5555.ini
Experiment config: /content/workspace/flexdc-sim/configs/experiment/new_iso/traditional_signal/generated_server_counts/exp_traditional_iso16_servers_1000.ini


## 10. Compile scripts — RUN

This checks syntax before running.


In [ ]:
%cd {TRAIN_DIR}
!python -m py_compile \
  am_unified_predict_one.py \
  am_unified_optimize_one.py \
  am_unified_end_to_end_eval_configured_objective.py


/content/workspace/comder-main/am_flexdc/train


## 11. Predict one point — OPTIONAL model-only test

Run this to check that the checkpoint can produce a prediction for the starting configuration. This does **not** run FlexDC.


In [ ]:
%cd {TRAIN_DIR}

PREDICT_OUT = RESULTS_DIR / f"predict_one_{TARGET_FAMILY}_{TARGET_MODE}.json"

!python am_unified_predict_one.py \
  --model-file "{MODEL_FILE}" \
  --norm-source-results-csv "{RESULTS_CSV}" \
  --workload-config "{WORKLOAD_CONFIG}" \
  --experiment-config "{EXPERIMENT_CONFIG}" \
  --target-family "{TARGET_FAMILY}" \
  --target-mode "{TARGET_MODE}" \
  --raw-qos-aggregation "{RAW_QOS_AGGREGATION}" \
  --use-norm-cost "{USE_NORM_COST}" \
  --use-norm-pr "{USE_NORM_PR}" \
  --server-count "{SERVER_COUNT}" \
  --utilization "{UTILIZATION}" \
  --pbar-kw-per-server "{START_PBAR}" \
  --r-kw-per-server "{START_R}" \
  --weights "{START_WEIGHTS}" \
  --objective-weights "{OBJECTIVE_WEIGHTS}" \
  --device "{DEVICE}" \
  --out-json "{PREDICT_OUT}" \
  --wandb-project "{WANDB_PROJECT}" \
  --wandb-entity "{WANDB_ENTITY}" \
  --wandb-run-name "predict-one-{TARGET_FAMILY}-{TARGET_MODE}" \
  --wandb-mode "{WANDB_MODE}"


/content/workspace/comder-main/am_flexdc/train
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: amenon06 (amenon06-boston-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run 90sg6wgl (0.0s)
wandb: ⣻ setting up run 90sg6wgl (0.0s)
wandb: ⣽ setting up run 90sg6wgl (0.0s)
wandb: ⣾ setting up run 90sg6wgl (0.0s)
wandb: ⣷ setting up run 90sg6wgl (0.5s)
wandb: ⣯ setting up run 90sg6wgl (0.5s)
wandb: ⣟ setting up run 90sg6wgl (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/workspace/comder-main/am_flexdc/train/wandb/run-20260629_201738-90sg6wgl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run predict-one-condor-normal
wandb: ⭐️ View project at https://wandb.ai/amenon06-boston-university/flexdc-unified-inference
wandb: 🚀 View run at https://wand

## 12. Optimize only — OPTIONAL dry run

Run this before full FlexDC validation if you only want the surrogate-selected candidate. This does **not** run FlexDC.


In [ ]:
%cd {TRAIN_DIR}

OPT_DIR = RESULTS_DIR / f"optimize_only_{TARGET_FAMILY}_{TARGET_MODE}"

!python am_unified_optimize_one.py \
  --model-file "{MODEL_FILE}" \
  --norm-source-results-csv "{RESULTS_CSV}" \
  --workload-config "{WORKLOAD_CONFIG}" \
  --experiment-config "{EXPERIMENT_CONFIG}" \
  --target-family "{TARGET_FAMILY}" \
  --target-mode "{TARGET_MODE}" \
  --raw-qos-aggregation "{RAW_QOS_AGGREGATION}" \
  --use-norm-cost "{USE_NORM_COST}" \
  --use-norm-pr "{USE_NORM_PR}" \
  --server-count "{SERVER_COUNT}" \
  --utilization "{UTILIZATION}" \
  --start-pbar-kw-per-server "{START_PBAR}" \
  --start-r-kw-per-server "{START_R}" \
  --start-weights "{START_WEIGHTS}" \
  --iterations "{ITERATIONS}" \
  --lr "{LR}" \
  --objective-weights "{OBJECTIVE_WEIGHTS}" \
  --device "{DEVICE}" \
  --out-dir "{OPT_DIR}" \
  --wandb-project "{WANDB_PROJECT}" \
  --wandb-entity "{WANDB_ENTITY}" \
  --wandb-run-name "optimize-one-{TARGET_FAMILY}-{TARGET_MODE}" \
  --wandb-mode "{WANDB_MODE}"


/content/workspace/comder-main/am_flexdc/train
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: amenon06 (amenon06-boston-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run ob0buj8o (0.0s)
wandb: ⣻ setting up run ob0buj8o (0.0s)
wandb: ⣽ setting up run ob0buj8o (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/workspace/comder-main/am_flexdc/train/wandb/run-20260629_201819-ob0buj8o
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run optimize-one-condor-normal
wandb: ⭐️ View project at https://wandb.ai/amenon06-boston-university/flexdc-unified-inference
wandb: 🚀 View run at https://wandb.ai/amenon06-boston-university/flexdc-unified-inference/runs/ob0buj8o
wandb: ⢿ updating run metadata (0.6s)
wandb: ⣻ updating run metadata (0.6s)
wandb: ⣽ upd

## 13. Full end-to-end evaluation — RUN when ready

This runs optimization, then runs FlexDC twice: starting configuration and selected configuration.

Skip this if the FlexDC repo/dependencies are not installed in the Colab runtime.


In [ ]:
%cd {TRAIN_DIR}

E2E_DIR = RESULTS_DIR / f"e2e_{TARGET_FAMILY}_{TARGET_MODE}_W2_N1000_U080"

!python am_unified_end_to_end_eval_configured_objective.py \
  --model-file "{MODEL_FILE}" \
  --norm-source-results-csv "{RESULTS_CSV}" \
  --workload-config "{WORKLOAD_CONFIG}" \
  --experiment-config "{EXPERIMENT_CONFIG}" \
  --target-family "{TARGET_FAMILY}" \
  --target-mode "{TARGET_MODE}" \
  --raw-qos-aggregation "{RAW_QOS_AGGREGATION}" \
  --use-norm-cost "{USE_NORM_COST}" \
  --use-norm-pr "{USE_NORM_PR}" \
  --server-count "{SERVER_COUNT}" \
  --utilization "{UTILIZATION}" \
  --start-pbar-kw-per-server "{START_PBAR}" \
  --start-r-kw-per-server "{START_R}" \
  --start-weights "{START_WEIGHTS}" \
  --iterations "{ITERATIONS}" \
  --lr "{LR}" \
  --objective-weights "{OBJECTIVE_WEIGHTS}" \
  --device "{DEVICE}" \
  --flexdc-root "{FLEXDC_ROOT}" \
  --flexdc-python python \
  --run-flexdc \
  --out-dir "{E2E_DIR}" \
  --wandb-project "{WANDB_PROJECT}" \
  --wandb-entity "{WANDB_ENTITY}" \
  --wandb-run-name "e2e-{TARGET_FAMILY}-{TARGET_MODE}-W2-N1000-U080" \
  --wandb-mode "{WANDB_MODE}"


/content/workspace/comder-main/am_flexdc/train
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: amenon06 (amenon06-boston-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run lwv398z1 (0.0s)
wandb: ⣻ setting up run lwv398z1 (0.0s)
wandb: ⣽ setting up run lwv398z1 (0.0s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/workspace/comder-main/am_flexdc/train/wandb/run-20260629_201835-lwv398z1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run e2e-condor-normal-W2-N1000-U080
wandb: ⭐️ View project at https://wandb.ai/amenon06-boston-university/flexdc-unified-inference
wandb: 🚀 View run at https://wandb.ai/amenon06-boston-university/flexdc-unified-inference/runs/lwv398z1

Running FlexDC start
python -u am_data_extraction_wizard.py --gradient-config ../.

## 14. Inspect and download outputs — RUN after evaluation


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Change this only if your orchestrator output folder has a different name.
# By default this uses E2E_DIR from the full end-to-end cell above.
if "EVAL_OUT_DIR" in globals():
    EVAL_OUT_DIR = Path(EVAL_OUT_DIR)
elif "E2E_DIR" in globals():
    EVAL_OUT_DIR = Path(E2E_DIR)
else:
    EVAL_OUT_DIR = Path("/content/workspace/comder-main/am_flexdc/results/unified_eval_runs/e2e_condor_normal_W2_N1000_U080")

summary_path = EVAL_OUT_DIR / "end_to_end_validation_summary.csv"
candidate_path = EVAL_OUT_DIR / "optimized_candidate.json"

summary = pd.read_csv(summary_path)

MAX_QOS_RATIO = 0.10
MAX_P90_TRACKING = 0.30

def find_row(df, keyword, fallback_idx):
    matches = df[df["Configuration"].str.contains(keyword, case=False, na=False, regex=True)]
    if len(matches) > 0:
        return matches.iloc[0]
    return df.iloc[fallback_idx]

start = find_row(summary, "Starting", 0)
selected = find_row(summary, "Selected|CONDOR-selected|FlexDC-selected", 1)

def val(row, col, default=np.nan):
    return row[col] if col in row.index else default

def fmt(x, digits=4):
    if pd.isna(x):
        return ""
    return f"{float(x):.{digits}f}"

def pct_change(start_value, selected_value):
    if pd.isna(start_value) or pd.isna(selected_value) or abs(float(start_value)) < 1e-12:
        return np.nan
    return 100.0 * (float(selected_value) - float(start_value)) / abs(float(start_value))

def change_text(start_value, selected_value, digits=4):
    if pd.isna(start_value) or pd.isna(selected_value):
        return ""
    diff = float(selected_value) - float(start_value)
    pct = pct_change(start_value, selected_value)
    if pd.isna(pct):
        return f"{diff:+.{digits}f}"
    return f"{diff:+.{digits}f} ({pct:+.2f}%)"

def lower_better_meaning(start_value, selected_value, good_text, bad_text, same_text="No meaningful change"):
    if pd.isna(start_value) or pd.isna(selected_value):
        return ""
    diff = float(selected_value) - float(start_value)
    if abs(diff) < 1e-9:
        return same_text
    return good_text if diff < 0 else bad_text

def qos_meaning(start_value, selected_value):
    if selected_value <= MAX_QOS_RATIO and start_value > MAX_QOS_RATIO:
        return "QoS fixed"
    if selected_value <= MAX_QOS_RATIO and start_value <= MAX_QOS_RATIO:
        return "QoS remained acceptable"
    if selected_value > MAX_QOS_RATIO and start_value <= MAX_QOS_RATIO:
        return "QoS became unacceptable"
    return "QoS still bad"

def tracking_meaning(start_value, selected_value):
    if selected_value <= MAX_P90_TRACKING and start_value <= MAX_P90_TRACKING:
        if selected_value <= start_value:
            return "Tracking acceptable and improved"
        return "Tracking acceptable but worse"
    if selected_value > MAX_P90_TRACKING and start_value <= MAX_P90_TRACKING:
        return "Tracking became unacceptable"
    if selected_value <= MAX_P90_TRACKING and start_value > MAX_P90_TRACKING:
        return "Tracking fixed"
    return "Tracking still unacceptable"

def add_row(rows, metric, start_value, selected_value, meaning):
    rows.append({
        "Metric": metric,
        "Starting": fmt(start_value),
        "Selected": fmt(selected_value),
        "Change": change_text(start_value, selected_value),
        "Meaning": meaning,
    })

rows = []

# Configuration rows
start_p = val(start, "Pbar_kw_per_server")
sel_p = val(selected, "Pbar_kw_per_server")
start_r = val(start, "R_kw_per_server")
sel_r = val(selected, "R_kw_per_server")

add_row(rows, "Pbar", start_p, sel_p, "Optimizer changed average power bid")
add_row(rows, "R", start_r, sel_r, "Optimizer changed reserve/flexibility bid")
add_row(rows, "Pbar + R", start_p + start_r, sel_p + sel_r, "Upper power-target bound")
add_row(rows, "Pbar - R", start_p - start_r, sel_p - sel_r, "Lower power-target bound")

# Objective rows
add_row(
    rows,
    "Predicted objective",
    val(start, "Predicted_Optimization_Objective"),
    val(selected, "Predicted_Optimization_Objective"),
    lower_better_meaning(
        val(start, "Predicted_Optimization_Objective"),
        val(selected, "Predicted_Optimization_Objective"),
        "Model expected improvement",
        "Model expected worse result",
    ),
)

add_row(
    rows,
    "Actual objective",
    val(start, "Actual_Optimization_Objective"),
    val(selected, "Actual_Optimization_Objective"),
    lower_better_meaning(
        val(start, "Actual_Optimization_Objective"),
        val(selected, "Actual_Optimization_Objective"),
        "FlexDC-Sim confirm objective improved",
        "FlexDC-Sim objective got worse",
    ),
)

# Diagnostics rows
add_row(
    rows,
    "QoS violation ratio",
    val(start, "QoS_Violation_Ratio"),
    val(selected, "QoS_Violation_Ratio"),
    qos_meaning(val(start, "QoS_Violation_Ratio"), val(selected, "QoS_Violation_Ratio")),
)

add_row(
    rows,
    "p90 tracking error",
    val(start, "Ctrack_Epsilon_90th"),
    val(selected, "Ctrack_Epsilon_90th"),
    tracking_meaning(val(start, "Ctrack_Epsilon_90th"), val(selected, "Ctrack_Epsilon_90th")),
)

add_row(
    rows,
    "FlexDC full objective",
    val(start, "Diagnostic_FullPaperObjective_Cost"),
    val(selected, "Diagnostic_FullPaperObjective_Cost"),
    lower_better_meaning(
        val(start, "Diagnostic_FullPaperObjective_Cost"),
        val(selected, "Diagnostic_FullPaperObjective_Cost"),
        "FlexDC full objective improved",
        "FlexDC full objective got worse",
    ),
)

report = pd.DataFrame(rows)

# Dark-mode readable table styling.
def style_row(row):
    base_bg = "#111827" if row.name % 2 else "#1f2937"
    base_style = (
        f"background-color: {base_bg}; "
        "color: #f9fafb; "
        "border: 1px solid #475569; "
        "padding: 7px; "
        "font-size: 13px; "
        "text-align: left; "
        "white-space: normal;"
    )

    styles = [base_style] * len(row)

    text = str(row["Meaning"]).lower()
    meaning_idx = row.index.get_loc("Meaning")

    good_words = ["fixed", "improved", "confirmed", "acceptable and improved", "remained acceptable"]
    bad_words = ["worse", "bad", "unacceptable", "violated"]

    if any(word in text for word in good_words):
        styles[meaning_idx] = (
            "background-color: #064e3b; "
            "color: #ecfdf5; "
            "font-weight: 800; "
            "border: 1px solid #34d399; "
            "padding: 7px; "
            "font-size: 13px; "
            "text-align: left; "
            "white-space: normal;"
        )
    elif any(word in text for word in bad_words):
        styles[meaning_idx] = (
            "background-color: #7f1d1d; "
            "color: #fef2f2; "
            "font-weight: 800; "
            "border: 1px solid #fca5a5; "
            "padding: 7px; "
            "font-size: 13px; "
            "text-align: left; "
            "white-space: normal;"
        )

    return styles

styled = (
    report.style
    .hide(axis="index")
    .apply(style_row, axis=1)
    .set_table_styles([
        {"selector": "th", "props": [
            ("background-color", "#0f172a"),
            ("color", "#ffffff"),
            ("font-weight", "800"),
            ("text-align", "left"),
            ("font-size", "13px"),
            ("border", "1px solid #475569"),
            ("padding", "8px"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("width", "100%"),
            ("table-layout", "auto"),
        ]},
    ])
)

display(Markdown("### Main comparison"))
display(styled)

# Save main report
report_csv = EVAL_OUT_DIR / "main_comparison_table.csv"
report_md = EVAL_OUT_DIR / "main_comparison_table.md"
report.to_csv(report_csv, index=False)
report.to_markdown(report_md, index=False)

print("Saved:", report_csv)
print("Saved:", report_md)

# -----------------------------
# Starting vs selected P/R/w
# -----------------------------
def format_weights(weights, digits=6):
    if weights is None or len(weights) == 0:
        return ""
    return "\n".join([f"w{i}={float(w):.{digits}f}" for i, w in enumerate(weights)])

candidate = {}
if candidate_path.exists():
    with open(candidate_path, "r") as f:
        candidate = json.load(f)

start_weights = candidate.get("starting_weights", None)
selected_weights = (
    candidate.get("optimized_weights", None)
    or candidate.get("selected_weights", None)
)

# Fallback: try summary column if present.
if start_weights is None and "Weights" in start.index:
    try:
        start_weights = json.loads(start["Weights"])
    except Exception:
        start_weights = []
if selected_weights is None and "Weights" in selected.index:
    try:
        selected_weights = json.loads(selected["Weights"])
    except Exception:
        selected_weights = []

config_table = pd.DataFrame([
    {
        "Configuration": "Starting",
        "Pbar": fmt(start_p, 6),
        "R": fmt(start_r, 6),
        "Pbar + R": fmt(start_p + start_r, 6),
        "Pbar - R": fmt(start_p - start_r, 6),
        "Weights": format_weights(start_weights),
    },
    {
        "Configuration": "Selected",
        "Pbar": fmt(sel_p, 6),
        "R": fmt(sel_r, 6),
        "Pbar + R": fmt(sel_p + sel_r, 6),
        "Pbar - R": fmt(sel_p - sel_r, 6),
        "Weights": format_weights(selected_weights),
    },
])

config_styled = (
    config_table.style
    .hide(axis="index")
    .set_properties(**{
        "background-color": "#111827",
        "color": "#f9fafb",
        "border": "1px solid #475569",
        "padding": "7px",
        "font-size": "13px",
        "text-align": "left",
        "white-space": "pre-wrap",
    })
    .set_table_styles([
        {"selector": "th", "props": [
            ("background-color", "#0f172a"),
            ("color", "#ffffff"),
            ("font-weight", "800"),
            ("text-align", "left"),
            ("font-size", "13px"),
            ("border", "1px solid #475569"),
            ("padding", "8px"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("width", "100%"),
            ("table-layout", "auto"),
        ]},
    ])
)

display(Markdown("### Starting vs selected configuration"))
display(config_styled)

config_csv = EVAL_OUT_DIR / "starting_vs_selected_config.csv"
config_md = EVAL_OUT_DIR / "starting_vs_selected_config.md"
config_table.to_csv(config_csv, index=False)
config_table.to_markdown(config_md, index=False)

print("Saved:", config_csv)
print("Saved:", config_md)


In [ ]:
import pandas as pd
from google.colab import files

if 'E2E_DIR' in globals() and E2E_DIR.exists():
    summary_path = E2E_DIR / "end_to_end_validation_summary.csv"
    report_path = E2E_DIR / "end_to_end_validation_report.md"
    if summary_path.exists():
        display(pd.read_csv(summary_path))
    if report_path.exists():
        print(report_path.read_text())

    zip_path = Path('/content') / f"{E2E_DIR.name}.zip"
    !zip -r "{zip_path}" "{E2E_DIR}"
    files.download(str(zip_path))
else:
    print("No E2E_DIR found yet. Run the full end-to-end cell first.")
